In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import nltk, string
from sklearn.feature_extraction.text import TfidfVectorizer



In [2]:
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/test.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/description.md
/kaggle/input/train.csv.zip
/kaggle/input/test.csv.zip
/kaggle/input/train.csv
/kaggle/input/sample_submission.csv
/kaggle/input/us-patent-phrase-to-phrase-matching/test.csv
/kaggle/input/us-patent-phrase-to-phrase-matching/sample_submission.csv.zip
/kaggle/input/us-patent-phrase-to-phrase-matching/description.md
/kaggle/input/us-patent-phrase-to-phrase-matching/train.csv.zip
/kaggle/input/us-patent-phrase-to-phrase-matching/test.csv.zip
/kaggle/input/us-patent-phrase-to-phrase-matching/train.csv
/kaggle/input/us-patent-phrase-to-phrase-matching/sample_submission.csv


In [3]:
train_path = "/kaggle/input/us-patent-phrase-to-phrase-matching/train.csv"
test_path = "/kaggle/input/us-patent-phrase-to-phrase-matching/test.csv"
sample_submission_path = (
    "/kaggle/input/us-patent-phrase-to-phrase-matching/sample_submission.csv"
)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(sample_submission_path)



In [4]:
target = train["score"]



In [5]:
train_features = train.drop(["score"], axis=1)
combi = pd.concat([train_features, test], ignore_index=True)



In [6]:
X = combi.iloc[: len(train)].reset_index(drop=True)
X_test = combi.iloc[len(train) :].reset_index(drop=True)



In [7]:
vectorizer = TfidfVectorizer(min_df=1, stop_words="english")


def cosine_sim(text1: str, text2: str) -> float:
    """
    Compute cosine similarity between two strings using TF‑IDF.
    The TF‑IDF matrix is converted to a dense array with `.toarray()`
    because the sparse matrix type does not expose the `.A` attribute.
    The resulting similarity is clipped to the [0, 1] range, matching
    the competition’s score bounds.
    """
    tfidf = vectorizer.fit_transform([text1, text2])
    # dense similarity matrix
    sim_matrix = (tfidf * tfidf.T).toarray()
    sim = sim_matrix[0, 1]
    # Clip to valid range
    return float(np.clip(sim, 0.0, 1.0))




In [8]:
simularity = []
for i in range(len(X_test)):
    anchor = str(X_test.loc[i, "anchor"]) + " " + str(X_test.loc[i, "context"])
    target_phrase = str(X_test.loc[i, "target"]) + " " + str(X_test.loc[i, "context"])
    sim = cosine_sim(anchor, target_phrase)
    simularity.append(sim)

assert len(simularity) == len(
    submission
), "Length mismatch between predictions and submission template."



In [9]:
submission["score"] = simularity
output_path = "/kaggle/working/submission.csv"
submission.to_csv(output_path, index=False)
print(f"Submission saved to {output_path}")

Submission saved to /kaggle/working/submission.csv
